
Note book Extract Script - Thu thập dữ liệu từ Binance API

Chức năng:
1. Extract Symbols: Lấy thông tin 50 coins từ /exchangeInfo, lưu thẳng vào data/raw 
2. [TEST] Extract Klines: Lấy dữ liệu nến 1 phút từ /klines
3. [TEST] Extract Ticker 24h + Best Bid/Ask: Lấy thống kê 24h từ /ticker/24hr và /ticker/bookTicker
4. [TEST] Extract Order Book Snapshot: Lấy snapshot từ /depth
(Test trước 10 symbol, lưu tạm tại notebooks/01_data_exploration/data/raw/)
Output:
   - data/raw/symbols.json
   - data/raw/{SYMBOL}.csv (50 files)
   - data/raw/ticker_24h.csv
   - data/raw/order_book_snapshot.csv

Sử dụng: python scripts/extract.py --start-date 2023-01-01 --end-date 2026-01-01


In [8]:
# 1. Extract Symbols: Lấy thông tin 50 coins từ /exchangeInfo:
import csv
import json
import os
from typing import List, Dict
import logging
from pathlib import Path

import requests

BASE_URL = "https://api.binance.com"

SYMBOLS: List[str] = [
    "BTCUSDT", "ETHUSDT", "BNBUSDT", "SOLUSDT", "XRPUSDT",
    "DOGEUSDT", "ADAUSDT", "TRXUSDT", "LINKUSDT", "MATICUSDT",
    "AVAXUSDT", "TONUSDT", "SHIBUSDT", "XLMUSDT", "BCHUSDT",
    "DOTUSDT", "UNIUSDT", "LTCUSDT", "HBARUSDT", "PEPEUSDT",
    "NEARUSDT", "APTUSDT", "ICPUSDT", "ETCUSDT", "STXUSDT",
    "RENDERUSDT", "CROUSDT", "ATOMUSDT", "VETUSDT", "ARBUSDT",
    "INJUSDT", "IMXUSDT", "OPUSDT", "GRTUSDT", "THETAUSDT",
    "FILUSDT", "ARUSDT", "MKRUSDT", "WIFUSDT", "RUNEUSDT",
    "FTMUSDT", "ALGOUSDT", "FLOWUSDT", "XTZUSDT", "AXSUSDT",
    "SANDUSDT", "MANAUSDT", "NEOUSDT", "EOSUSDT", "AAVEUSDT",
]

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

def _parse_symbols(payload: dict) -> List[Dict[str, str]]:
    results = []
    for item in payload.get("symbols", []):
        results.append(
            {
                "symbol": item.get("symbol"),
                "base_asset": item.get("baseAsset"),
                "quote_asset": item.get("quoteAsset"),
                "status": item.get("status"),
            }
        )
    return results

def get_exchange_info(symbols: List[str]) -> List[Dict[str, str]]:
    params = {"symbols": json.dumps(symbols)}
    try:
        response = requests.get(f"{BASE_URL}/api/v3/exchangeInfo", params=params, timeout=30)
        response.raise_for_status()
        return _parse_symbols(response.json())
    except requests.HTTPError as exc:
        if exc.response is not None and exc.response.status_code == 400:
            full = requests.get(f"{BASE_URL}/api/v3/exchangeInfo", timeout=30)
            full.raise_for_status()
            payload = full.json()
            valid = {item.get("symbol") for item in payload.get("symbols", [])}
            filtered = [s for s in symbols if s in valid]
            return [item for item in _parse_symbols(payload) if item.get("symbol") in set(filtered)]
        raise

def save_csv(rows: List[Dict[str, str]], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["symbol", "base_asset", "quote_asset", "status"])
        writer.writeheader()
        writer.writerows(rows)

def save_json(rows: List[Dict[str, str]], path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(rows, f, ensure_ascii=False, indent=2)

def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for _ in range(10):
        if (current / "data" / "raw").exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    return start.resolve()

def main() -> None:
    rows = get_exchange_info(SYMBOLS)

    start_dir = Path.cwd()
    project_root = find_project_root(start_dir)
    raw_dir = project_root / "data" / "raw"

    csv_path = raw_dir / "symbols.csv"
    json_path = raw_dir / "symbols.json"

    save_csv(rows, str(csv_path))
    save_json(rows, str(json_path))

    found = {r.get("symbol") for r in rows}
    expected = set(SYMBOLS)
    missing = sorted(expected - found)

    logger.info("Saved %s symbols to: %s", len(rows), csv_path)
    logger.info("Saved %s symbols to: %s", len(rows), json_path)
    if missing:
        logger.warning("Missing symbols: %s", ", ".join(missing))
    else:
        logger.info("All symbols returned.")

if __name__ == "__main__":
    main()

INFO: Saved 49 symbols to: /home/wuudes/projects/Crypto-Data-Pipeline/data/raw/symbols.csv
INFO: Saved 49 symbols to: /home/wuudes/projects/Crypto-Data-Pipeline/data/raw/symbols.json
